# Kraków - oferty mieszkań 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Matix732/wizualizacja-danych-uj/blob/main/Kraków_mieszkania_mapa.ipynb)

Notebook korzysta z pliku `Otodom_Flat_Listings.csv` z repozytorium GitHub.

Zakres:
- filtrujemy oferty do Krakowa,
- wyciągamy dzielnice i poddzielnice oraz cechy mieszkań,
- robimy mapę dzielnic i wizualizacje danych,

In [ ]:
# Google Colab: instalacja bibliotek
%pip -q install geopandas folium osmnx geopy squarify circlify adjustText

In [ ]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import osmnx as ox
import circlify

from pathlib import Path
from matplotlib.ticker import FuncFormatter
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

import matplotlib.patheffects as path_effects
from adjustText import adjust_text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
print("Importy gotowe")

Importy gotowe


In [ ]:
# Wczytanie danych z repozytorium GitHub
import urllib.request

csv_url = "https://raw.githubusercontent.com/Matix732/wizualizacja-danych-uj/main/Otodom_Flat_Listings.csv"

try:
    df = pd.read_csv(csv_url)
    print(f"✓ Wczytano z GitHub: {csv_url}")
except Exception as e:
    print(f"✗ Błąd przy pobieraniu z GitHub: {e}")
    raise

print(f"Liczba rekordów (cała Polska): {len(df):,}")

# Normalizacja kolumn
rename_map = {
    "Title": "title",
    "Price": "price",
    "Location": "location",
    "Surface": "surface",
    "Number_of_Rooms": "rooms",
    "Link": "link",
    "City": "city",
}
df = df.rename(columns=rename_map)

for col in ["title", "price", "location", "surface", "rooms", "link", "city"]:
    if col not in df.columns:
        df[col] = np.nan

df["city_norm"] = df["city"].astype(str).str.lower().str.strip()
df["location_norm"] = df["location"].astype(str).str.lower().str.strip()

# Zostawiamy tylko Kraków (na podstawie city lub location)
krk = df[
    df["city_norm"].str.contains("krak", na=False)
    | df["location_norm"].str.contains("krak", na=False)
].copy()

krk["price"] = pd.to_numeric(krk["price"], errors="coerce")
krk["surface"] = pd.to_numeric(krk["surface"], errors="coerce")
krk["rooms"] = pd.to_numeric(krk["rooms"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
krk = krk.dropna(subset=["price"])
krk = krk[krk["price"].between(50_000, 10_000_000)]

print(f"Oferty w Krakowie: {len(krk):,}")
krk[["title", "price", "location", "city"]].head(5)

FileNotFoundError: Nie znaleziono pliku CSV. Umieść go w /content lub katalogu roboczym.

In [ ]:
DISTRICTS = [
    "Stare Miasto", "Grzegórzki", "Prądnik Czerwony", "Prądnik Biały", "Krowodrza",
    "Bronowice", "Zwierzyniec", "Dębniki", "Łagiewniki-Borek Fałęcki", "Swoszowice",
    "Podgórze Duchackie", "Bieżanów-Prokocim", "Podgórze", "Czyżyny", "Mistrzejowice",
    "Bieńczyce", "Wzgórza Krzesławickie", "Nowa Huta"
]

district_keywords = {
    d: [d.lower()] for d in DISTRICTS
}
district_keywords["Stare Miasto"] += ["kazimierz", "stradom", "stare miasto"]
district_keywords["Podgórze"] += ["zabłocie", "podgorze"]
district_keywords["Krowodrza"] += ["azory"]
district_keywords["Dębniki"] += ["ruczaj", "debniki"]
district_keywords["Nowa Huta"] += ["nowa huta"]

def extract_street(text: str):
    if not isinstance(text, str):
        return None
    m = re.search(r"(?:ul\.?|al\.?|os\.?)[\s]+([A-ZĄĆĘŁŃÓŚŹŻa-ząćęłńóśźż0-9\- ]{3,})", text)
    return m.group(0).strip() if m else None

def normalize_tokens(location: str):
    if not isinstance(location, str):
        return []
    return [t.strip() for t in location.split(",") if t.strip()]

def guess_district_from_text(text: str):
    if not isinstance(text, str):
        return None
    txt = text.lower()
    for district, keys in district_keywords.items():
        if any(k in txt for k in keys):
            return district
    return None

def extract_subdistrict(location: str, district: str):
    tokens = normalize_tokens(location)
    if not tokens:
        return None
    cleaned = []
    for t in tokens:
        tl = t.lower()
        if "krak" in tl or "małopol" in tl or tl in {"polska", "poland"}:
            continue
        if district and district.lower() in tl:
            continue
        cleaned.append(t)
    return cleaned[0] if cleaned else None

krk["street"] = krk["location"].apply(extract_street)
krk.loc[krk["street"].isna(), "street"] = krk.loc[krk["street"].isna(), "title"].apply(extract_street)

krk["district"] = krk["location"].apply(guess_district_from_text)
krk["subdistrict"] = krk.apply(lambda r: extract_subdistrict(r["location"], r["district"]), axis=1)

# Wymaganie 4: jeśli brak lokalizacji, użyj ulicy z tytułu i geokoduj do dzielnicy
missing_loc = krk[krk["district"].isna() & krk["street"].notna()].copy()
print(f"Brak dzielnicy po samym location: {len(missing_loc)} rekordów")

if len(missing_loc) > 0:
    geolocator = Nominatim(user_agent="krakow_district_lookup_colab")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

    def district_from_street(street):
        try:
            loc = geocode(f"{street}, Kraków, Polska", addressdetails=True)
            if not loc:
                return None, None
            rev = reverse((loc.latitude, loc.longitude), language="pl", addressdetails=True)
            addr = rev.raw.get("address", {}) if rev else {}
            district_text = " ".join([
                str(addr.get("city_district", "")),
                str(addr.get("suburb", "")),
                str(addr.get("neighbourhood", "")),
            ]).strip()
            district = guess_district_from_text(district_text)
            subdistrict = addr.get("neighbourhood") or addr.get("suburb") or addr.get("quarter")
            return district, subdistrict
        except Exception:
            return None, None

    for idx, row in missing_loc.iterrows():
        d, s = district_from_street(row["street"])
        if d:
            krk.at[idx, "district"] = d
        if s and pd.isna(krk.at[idx, "subdistrict"]):
            krk.at[idx, "subdistrict"] = s

# Domknięcie braków
krk["district"] = krk["district"].fillna("Nieustalona dzielnica")
krk["subdistrict"] = krk["subdistrict"].fillna("Nieustalona poddzielnica")
krk["price_m2"] = (krk["price"] / krk["surface"]).replace([np.inf, -np.inf], np.nan)

krk[["title", "street", "district", "subdistrict", "price", "surface"]].head(10)

In [ ]:
# Granice dzielnic Krakowa (GeoPandas + OSM)
district_gdfs = []
for district in DISTRICTS:
    try:
        q = f"{district}, Kraków, województwo małopolskie, Polska"
        g = ox.geocode_to_gdf(q)
        g["district"] = district
        district_gdfs.append(g[["district", "geometry"]])
    except Exception:
        pass

if not district_gdfs:
    raise RuntimeError("Nie udało się pobrać granic dzielnic z OSM.")

districts_gdf = pd.concat(district_gdfs, ignore_index=True)
districts_gdf = gpd.GeoDataFrame(districts_gdf, geometry="geometry", crs="EPSG:4326")
districts_gdf = districts_gdf.dissolve(by="district", as_index=False)

agg_district = (
    krk.groupby("district", dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
)

districts_plot = districts_gdf.merge(agg_district, on="district", how="left")
districts_plot[["listings", "median_price", "median_price_m2"]] = districts_plot[["listings", "median_price", "median_price_m2"]].fillna(0)

print("Dzielnice z geometrią:", len(districts_plot))
districts_plot.head(3)

In [ ]:
# Mapa dzielnic: liczba ofert (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="listings",
    cmap="YlOrRd",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

texts = []
for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        txt = ax.text(p.x, p.y, f"{row['district']}\n{int(row['listings'])}", 
                      fontsize=8, ha="center", color="black", weight="bold")
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])
        texts.append(txt)

adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='black', alpha=0.5))

ax.set_title("Kraków - liczba ofert mieszkań wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Mapa dzielnic: mediana ceny za m2 (choropleth)
fig, ax = plt.subplots(1, 1, figsize=(11, 11))
districts_plot.plot(
    column="median_price_m2",
    cmap="RdYlGn_r",
    linewidth=1,
    edgecolor="white",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey", "label": "Brak danych"},
)

texts = []
for _, row in districts_plot.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        p = row.geometry.representative_point()
        price_m2 = row['median_price_m2']
        txt = ax.text(p.x, p.y, f"{row['district']}\n{price_m2:,.0f} PLN/m²", 
                      fontsize=8, ha="center", color="black", weight="bold")
        txt.set_path_effects([path_effects.withStroke(linewidth=3, foreground='white')])
        texts.append(txt)

adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='black', alpha=0.5))

ax.set_title("Kraków - mediana ceny za m² wg dzielnic", fontsize=14, weight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# A) Grouped barplot: cena mieszkań w dzielnicach (min, średnia, max)
price_stats = (
    krk.dropna(subset=["district", "price"])
    .groupby("district", as_index=False)["price"]
    .agg(min_price="min", mean_price="mean", max_price="max")
)

price_long = price_stats.melt(
    id_vars="district",
    value_vars=["min_price", "mean_price", "max_price"],
    var_name="stat",
    value_name="value",
)

price_label_map = {
    "min_price": "Min",
    "mean_price": "Srednia",
    "max_price": "Max",
}
price_long["stat"] = price_long["stat"].map(price_label_map)

fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(
    data=price_long,
    x="district",
    y="value",
    hue="stat",
    palette=["#74c476", "#6baed6", "#fb6a4a"],
    ax=ax,
)
ax.set_title("Cena mieszkan wg dzielnic: Min / Srednia / Max", fontsize=14, weight="bold")
ax.set_xlabel("Dzielnica")
ax.set_ylabel("Cena [PLN]")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}".replace(",", " ")))
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Statystyka")
plt.tight_layout()
plt.show()

# B) Grouped barplot: metraz mieszkan w dzielnicach (min, średnia, max)
surface_stats = (
    krk.dropna(subset=["district", "surface"])
    .groupby("district", as_index=False)["surface"]
    .agg(min_surface="min", mean_surface="mean", max_surface="max")
)

surface_long = surface_stats.melt(
    id_vars="district",
    value_vars=["min_surface", "mean_surface", "max_surface"],
    var_name="stat",
    value_name="value",
)

surface_label_map = {
    "min_surface": "Min",
    "mean_surface": "Srednia",
    "max_surface": "Max",
}
surface_long["stat"] = surface_long["stat"].map(surface_label_map)

fig, ax = plt.subplots(figsize=(16, 7))
sns.barplot(
    data=surface_long,
    x="district",
    y="value",
    hue="stat",
    palette=["#74c476", "#6baed6", "#fb6a4a"],
    ax=ax,
)
ax.set_title("Metraz mieszkan wg dzielnic: Min / Srednia / Max", fontsize=14, weight="bold")
ax.set_xlabel("Dzielnica")
ax.set_ylabel("Metraz [m2]")
ax.tick_params(axis="x", rotation=45)
ax.legend(title="Statystyka")
plt.tight_layout()
plt.show()

In [ ]:
# Styl: Python Graph Gallery - Circular Barplot (mediana ceny po dzielnicach)
circular_data = (
    krk.groupby("district", dropna=False)["price"]
    .median()
    .sort_values(ascending=False)
    .reset_index(name="median_price")
)

N = len(circular_data)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
values = circular_data["median_price"].tolist()
labels = circular_data["district"].tolist()

fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection="polar"))
ax.bar(angles, values, width=0.5, alpha=0.85, color=sns.color_palette("RdYlGn_r", N))
ax.set_xticks(angles)
ax.set_xticklabels(labels, size=9)
ax.set_ylim(0, max(values) * 1.1)
ax.set_title("Circular Barplot: mediana ceny ofert wg dzielnic", fontsize=14, weight="bold", pad=20)
ax.grid(True, alpha=0.3)
plt.show()

# Circular Packing (one level): same kola, rozmiar zalezy od liczby ofert
packing_data = (
    krk.groupby("district", dropna=False)
    .size()
    .reset_index(name="listings")
    .sort_values("listings", ascending=False)
)

circles = circlify.circlify(
    packing_data["listings"].tolist(),
    show_enclosure=False,
    target_enclosure=circlify.Circle(x=0, y=0, r=1),
)

fig, ax = plt.subplots(figsize=(12, 12))
ax.set_title("Circular Packing: udzial dzielnic w liczbie ofert", fontsize=14, weight="bold")
ax.axis("off")
ax.set_aspect("equal")

# Dynamiczne dopasowanie granic wykresu żeby nie obcinało kółek
maxx = max(circle.x + circle.r for circle in circles)
minx = min(circle.x - circle.r for circle in circles)
maxy = max(circle.y + circle.r for circle in circles)
miny = min(circle.y - circle.r for circle in circles)
ax.set_xlim(minx - 0.1, maxx + 0.1)
ax.set_ylim(miny - 0.1, maxy + 0.1)

palette = sns.color_palette("Spectral", len(circles))
total = packing_data["listings"].sum()

for i, (circle, (_, row)) in enumerate(zip(circles, packing_data.iterrows())):
    x, y, r = circle.x, circle.y, circle.r
    share = row["listings"] / total * 100
    patch = plt.Circle((x, y), r, color=palette[i], alpha=0.88, ec="white", lw=1.5)
    ax.add_patch(patch)

    # Pokazuj napisy tylko na wystarczająco dużych kołach
    if r > (maxx - minx) * 0.05:  
        ax.text(
            x,
            y,
            f"{row['district']}\n{share:.1f}%",
            ha="center",
            va="center",
            fontsize=8,
            weight="bold",
        )

plt.tight_layout()
plt.show()

In [ ]:
# Lollipop chart: mediana ceny za 1 pokój w danej dzielnicy
krk["price_per_room"] = krk["price"] / krk["rooms"].replace(0, np.nan)
room_price_data = (
    krk.dropna(subset=["district", "price_per_room"])
    .groupby("district")["price_per_room"]
    .median()
    .sort_values()
    .reset_index(name="median_room_price")
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y=room_price_data["district"], xmin=0, xmax=room_price_data["median_room_price"], color="#e07a5f", linewidth=3)
ax.plot(room_price_data["median_room_price"], room_price_data["district"], "o", color="#de2d26", markersize=8)
ax.set_title("Lollipop: Mediana ceny za 1 pokój wg dzielnicy", fontsize=14, weight="bold")
ax.set_xlabel("Mediana ceny za pokój [PLN]")
ax.set_ylabel("Dzielnica")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}".replace(",", " ")))
ax.grid(axis="y", linestyle="")
plt.tight_layout()
plt.show()

# Lollipop chart: średnia liczba pokoi w danej dzielnicy
# (Stosujemy średnią, ponieważ mediana prawie wszędzie wynosiłaby 2 lub 3, co dałoby płaski wykres)
rooms_data = (
    krk.dropna(subset=["district", "rooms"])
    .groupby("district")["rooms"]
    .mean()
    .sort_values()
    .reset_index(name="mean_rooms")
)

fig, ax = plt.subplots(figsize=(10, 8))
ax.hlines(y=rooms_data["district"], xmin=0, xmax=rooms_data["mean_rooms"], color="#a6bddb", linewidth=3)
ax.plot(rooms_data["mean_rooms"], rooms_data["district"], "o", color="#02818a", markersize=8)
ax.set_title("Lollipop: Średnia liczba pokoi w mieszkaniu wg dzielnicy", fontsize=14, weight="bold")
ax.set_xlabel("Średnia liczba pokoi")
ax.set_ylabel("Dzielnica")
ax.grid(axis="y", linestyle="")
plt.tight_layout()
plt.show()

In [ ]:
# Top oferty wrzucone jako scatter plot - "najlepsza cena do powierzchni"
top_offers = krk.dropna(subset=["surface", "price"]).copy()

fig, ax = plt.subplots(figsize=(12, 7))

sns.scatterplot(
    data=top_offers,
    x="surface",
    y="price",
    hue="district",
    alpha=0.6,
    s=50,
    ax=ax,
    legend=False # Zbyt wiele dzielnic zeby czytelnie zmiescic legendę
)

# Dodatkowa linia obrazująca średnią cenę za m2 dla całego miasta
mean_m2_price = top_offers["price"].sum() / top_offers["surface"].sum()
x_vals = np.array([top_offers["surface"].min(), top_offers["surface"].max()])
ax.plot(x_vals, x_vals * mean_m2_price, color="red", linestyle="--", linewidth=2, 
        label=f"Średnia dla miasta: {mean_m2_price:,.0f} PLN/m2")

ax.set_title("Wykres z metrażem i końcową ceną ze wszystkich ofert", fontsize=14, weight="bold")
ax.set_xlabel("Powierzchnia [m2]")
ax.set_ylabel("Cena całkowita [PLN]")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(x):,}".replace(",", " ")))
ax.legend()
plt.tight_layout()
plt.show()

# Podsumowanie z exportem do CSV
summary = (
    krk.groupby(["district", "subdistrict"], dropna=False)
    .agg(
        listings=("title", "count"),
        median_price=("price", "median"),
        median_surface=("surface", "median"),
        median_price_m2=("price_m2", "median"),
    )
    .reset_index()
    .sort_values(["listings", "median_price"], ascending=[False, False])
)

print("Top 20 dzielnica/poddzielnica po liczbie ofert:")
display(summary.head(20))

out_path = Path("/content/krakow_offers_enriched.csv")
krk.to_csv(out_path, index=False)
print(f"Zapisano dane po czyszczeniu: {out_path}")